
##### 03 - Gold Layer Aggregation (Star Schema)

The Gold layer contains business-ready, analytics-optimized tables
modeled as a **star schema** with:
- Dimension tables: `dim_customers`, `dim_products`, `dim_date`
- Fact table: `fact_orders` (grain: one row per order line item)
- Aggregate tables: daily revenue, customer lifetime value, product performance

**Optimizations applied:**
- Partitioning on date columns
- Z-Ordering on high-cardinality filter columns
- File compaction via OPTIMIZE


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

##### 1. Dimension: dim_date
A complete date dimension covering the full data range.

In [0]:
df_date = (
    spark.sql("""
        SELECT explode(sequence(
            to_date('2025-01-01'),
            to_date('2025-12-31'),
            interval 1 day
        )) AS date_key
    """)
    .withColumn("year", F.year("date_key"))
    .withColumn("quarter", F.quarter("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("month_name", F.date_format("date_key", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("date_key"))
    .withColumn("day_of_month", F.dayofmonth("date_key"))
    .withColumn("day_of_week", F.dayofweek("date_key"))
    .withColumn("day_name", F.date_format("date_key", "EEEE"))
    .withColumn("is_weekend", F.when(F.dayofweek("date_key").isin(1, 7), True).otherwise(False))
    .withColumn("fiscal_quarter", F.concat(F.lit("FY25-Q"), F.quarter("date_key").cast("string")))
)

(
    df_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_gold.dim_date")
)

print(f"dim_date: {df_date.count()} rows")
df_date.show(5, truncate=False)

dim_date: 365 rows
+----------+----+-------+-----+----------+------------+------------+-----------+---------+----------+--------------+
|date_key  |year|quarter|month|month_name|week_of_year|day_of_month|day_of_week|day_name |is_weekend|fiscal_quarter|
+----------+----+-------+-----+----------+------------+------------+-----------+---------+----------+--------------+
|2025-01-01|2025|1      |1    |January   |1           |1           |4          |Wednesday|false     |FY25-Q1       |
|2025-01-02|2025|1      |1    |January   |1           |2           |5          |Thursday |false     |FY25-Q1       |
|2025-01-03|2025|1      |1    |January   |1           |3           |6          |Friday   |false     |FY25-Q1       |
|2025-01-04|2025|1      |1    |January   |1           |4           |7          |Saturday |true      |FY25-Q1       |
|2025-01-05|2025|1      |1    |January   |1           |5           |1          |Sunday   |true      |FY25-Q1       |
+----------+----+-------+-----+----------+---

##### 2. Dimension: dim_customers

In [0]:
df_customers = spark.table("ecommerce_silver.customers")
df_orders = spark.table("ecommerce_silver.orders")

customer_order_stats = (
    df_orders
    .groupBy("customer_id")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("total_amount").alias("total_spend"),
        F.avg("total_amount").alias("avg_order_value"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date"),
    )
)

df_dim_customers = (
    df_customers
    .join(customer_order_stats, "customer_id", "left")

    .withColumn("total_orders", F.coalesce(F.col("total_orders"), F.lit(0)))
    .withColumn("total_spend", F.coalesce(F.col("total_spend"), F.lit(0.0)))

    .withColumn("customer_segment",
                F.when(F.col("total_spend") >= 50000, "Platinum")
                .when(F.col("total_spend") >= 20000, "Gold")
                .when(F.col("total_spend") >= 5000, "Silver")
                .when(F.col("total_orders") >= 1, "Bronze")
                .otherwise("Prospect"))

    .withColumn("days_since_last_order",
                F.datediff(F.current_date(), F.col("last_order_date")))

    .select(
        "customer_id", "full_name", "email", "phone", "age",
        "city", "state", "signup_date", "status",
        "total_orders", "total_spend", "avg_order_value",
        "first_order_date", "last_order_date", "days_since_last_order",
        "customer_segment"
    )
)

(
    df_dim_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_gold.dim_customers")
)

spark.sql("OPTIMIZE ecommerce_gold.dim_customers ZORDER BY (city, customer_segment)")

print(f"dim_customers: {df_dim_customers.count()} rows")
print("\nCustomer Segment Distribution:")
df_dim_customers.groupBy("customer_segment").count().orderBy(F.desc("count")).show()


dim_customers: 10000 rows

Customer Segment Distribution:
+----------------+-----+
|customer_segment|count|
+----------------+-----+
|        Platinum| 5754|
|            Gold| 2223|
|          Silver| 1525|
|          Bronze|  446|
|        Prospect|   52|
+----------------+-----+



##### 3. Dimension: dim_products

In [0]:
df_products = spark.table("ecommerce_silver.products")
df_items = spark.table("ecommerce_silver.order_items")

product_sales_stats = (
    df_items
    .groupBy("product_id")
    .agg(
        F.sum("quantity").alias("total_units_sold"),
        F.sum("line_total").alias("total_revenue"),
        F.countDistinct("order_id").alias("order_count"),
        F.avg("discount_pct").alias("avg_discount"),
    )
)

df_dim_products = (
    df_products
    .join(product_sales_stats, "product_id", "left")
    .withColumn("total_units_sold", F.coalesce(F.col("total_units_sold"), F.lit(0)))
    .withColumn("total_revenue", F.coalesce(F.col("total_revenue"), F.lit(0.0)))
    .withColumn("revenue_rank",
                F.dense_rank().over(Window.orderBy(F.desc("total_revenue"))))
    .select(
        "product_id", "product_name", "category", "price", "price_tier",
        "rating", "stock_quantity", "status",
        "total_units_sold", "total_revenue", "order_count",
        "avg_discount", "revenue_rank"
    )
)

(
    df_dim_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_gold.dim_products")
)

spark.sql("OPTIMIZE ecommerce_gold.dim_products ZORDER BY (category)")

print(f"dim_products: {df_dim_products.count()} rows")
print("\nTop 10 Products by Revenue:")
df_dim_products.orderBy("revenue_rank").select(
    "revenue_rank", "product_name", "category", "total_revenue", "total_units_sold"
).show(10, truncate=False)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_products: 200 rows

Top 10 Products by Revenue:
+------------+------------+-----------+--------------------+----------------+
|revenue_rank|product_name|category   |total_revenue       |total_units_sold|
+------------+------------+-----------+--------------------+----------------+
|1           |Laptop      |Electronics|8.931146435000029E7 |725             |
|2           |Laptop      |Electronics|7.026732944000003E7 |744             |
|3           |Laptop      |Electronics|5.046565289000012E7 |813             |
|4           |Laptop      |Electronics|4.6544709930000044E7|785             |
|5           |Laptop      |Electronics|4.648295455999993E7 |863             |
|6           |Smartphone  |Electronics|4.4894109330000155E7|831             |
|7           |Tablet      |Electronics|4.2575772460000135E7|803             |
|8           |Smartphone  |Electronics|3.5066913659999944E7|769             |
|9           |Smartphone  |Electronics|2.9847648849999942E7|714             |
|10         

##### 4. Fact: fact_orders

In [0]:
df_orders = spark.table("ecommerce_silver.orders")
df_items = spark.table("ecommerce_silver.order_items")

df_fact_orders = (
    df_items
    .join(df_orders, "order_id", "inner")
    .select(
        "item_id",
        "order_id",
        "customer_id",
        "product_id",
        F.col("order_date").alias("order_date_key"),
        "order_year",
        "order_month",
        "status",
        "payment_method",
        "quantity",
        "unit_price",
        "discount_pct",
        "line_total",
        "shipping_fee",
    )
)

(
    df_fact_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("order_year", "order_month")
    .saveAsTable("ecommerce_gold.fact_orders")
)

spark.sql("OPTIMIZE ecommerce_gold.fact_orders ZORDER BY (customer_id, product_id)")

print(f"fact_orders: {df_fact_orders.count()} rows")
print(f"Partitions: year x month")


fact_orders: 105115 rows
Partitions: year x month


##### 5. Aggregate: Daily Revenue Summary

In [0]:
df_daily_revenue = (
    df_fact_orders
    .groupBy("order_date_key", "order_year", "order_month")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.sum("line_total").alias("gross_revenue"),
        F.sum("quantity").alias("total_units"),
        F.avg("line_total").alias("avg_line_value"),
        F.sum(F.when(F.col("status") == "cancelled", F.col("line_total")).otherwise(0)).alias("cancelled_revenue"),
    )
    .withColumn("net_revenue", F.col("gross_revenue") - F.col("cancelled_revenue"))
    .withColumn("revenue_7d_avg",
                F.avg("net_revenue").over(
                    Window.orderBy("order_date_key").rowsBetween(-6, 0)
                ))
    .orderBy("order_date_key")
)

(
    df_daily_revenue.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_gold.agg_daily_revenue")
)

print(f"agg_daily_revenue: {df_daily_revenue.count()} rows")
df_daily_revenue.show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


agg_daily_revenue: 365 rows
+--------------+----------+-----------+------------+----------------+------------------+-----------+------------------+------------------+------------------+------------------+
|order_date_key|order_year|order_month|total_orders|unique_customers|gross_revenue     |total_units|avg_line_value    |cancelled_revenue |net_revenue       |revenue_7d_avg    |
+--------------+----------+-----------+------------+----------------+------------------+-----------+------------------+------------------+------------------+------------------+
|2025-01-01    |2025      |1          |167         |165             |3462250.030000001 |539        |9459.699535519128 |322325.33000000013|3139924.700000001 |3139924.700000001 |
|2025-01-02    |2025      |1          |146         |145             |2736266.4400000004|518        |8217.016336336337 |544147.0600000003 |2192119.38        |2666022.0400000005|
|2025-01-03    |2025      |1          |127         |126             |1959140.4900000007

##### 6. Aggregate: Customer Lifetime Value (CLV)

In [0]:
df_clv = (
    df_fact_orders
    .filter(F.col("status") != "cancelled")
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("num_orders"),
        F.sum("line_total").alias("total_revenue"),
        F.avg("line_total").alias("avg_item_value"),
        F.min("order_date_key").alias("first_purchase"),
        F.max("order_date_key").alias("last_purchase"),
        F.countDistinct("product_id").alias("unique_products"),
    )
    .withColumn("customer_tenure_days",
                F.datediff(F.col("last_purchase"), F.col("first_purchase")))
    .withColumn("purchase_frequency",
                F.when(F.col("customer_tenure_days") > 0,
                       F.col("num_orders") / (F.col("customer_tenure_days") / 30.0))
                .otherwise(F.col("num_orders")))
    .withColumn("clv_score",
                F.round(F.col("total_revenue") * F.col("purchase_frequency") / F.greatest(F.col("num_orders"), F.lit(1)), 2))
    .withColumn("clv_tier",
                F.when(F.col("clv_score") >= 50000, "High")
                .when(F.col("clv_score") >= 10000, "Medium")
                .otherwise("Low"))
)

(
    df_clv.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_gold.agg_customer_clv")
)

print(f"agg_customer_clv: {df_clv.count()} rows")
print("\nCLV Tier Distribution:")
df_clv.groupBy("clv_tier").agg(
    F.count("*").alias("customers"),
    F.round(F.avg("total_revenue"), 2).alias("avg_revenue"),
    F.round(F.avg("num_orders"), 1).alias("avg_orders"),
).show()


agg_customer_clv: 9894 rows

CLV Tier Distribution:
+--------+---------+-----------+----------+
|clv_tier|customers|avg_revenue|avg_orders|
+--------+---------+-----------+----------+
|  Medium|     3912|  141918.07|       5.1|
|     Low|     5575|   32606.99|       4.1|
|    High|      407|  225049.07|       3.5|
+--------+---------+-----------+----------+



##### 7. Gold Layer Summary

In [0]:
gold_tables = ["dim_date", "dim_customers", "dim_products", "fact_orders",
               "agg_daily_revenue", "agg_customer_clv"]

for table in gold_tables:
    count = spark.table(f"ecommerce_gold.{table}").count()
    print(f"  ecommerce_gold.{table}: {count:>10,} rows")

print("\nGold layer complete! Proceed to notebook 04_data_quality_framework.")

  ecommerce_gold.dim_date:        365 rows
  ecommerce_gold.dim_customers:     10,000 rows
  ecommerce_gold.dim_products:        200 rows
  ecommerce_gold.fact_orders:    105,115 rows
  ecommerce_gold.agg_daily_revenue:        365 rows
  ecommerce_gold.agg_customer_clv:      9,894 rows

Gold layer complete! Proceed to notebook 04_data_quality_framework.
